# 12.01 · Production readiness: SLOs, error budgets, latency and cost budgets

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Module 10 (traces, intervals) and Project 09 (the Orbita Cloud support desk). Read the module README first. |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 TTFT probe) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **List** what separates a notebook agent from a production service, and score an agent against that checklist.
2. **Define** SLIs and SLOs for an agent (availability, time to first token, run duration) and evaluate them with 95% intervals.
3. **Use** an error budget: compute what is left, the burn rate during an incident, and the release policy it implies.
4. **Split** a latency SLO into a per-stage budget and find the stage that breaks it, at the right percentile.
5. **Enforce** a cost budget before a run starts: per request, per user per day, and a global kill switch.
6. **Review** failure modes (what fails, what the user sees, how you detect it, what limits the damage) before writing service code.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Implementing SLOs](https://sre.google/workbook/implementing-slos/) (Google SRE workbook) | SLIs as good events ÷ valid events, choosing targets, error budgets |
| [Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) | Burn rate and multi-window alerts (the 14.4× one-hour page) |
| [Error budget policy](https://sre.google/workbook/error-budget-policy/) | What a team agrees to do when the budget runs out |
| [Groq rate limits](https://console.groq.com/docs/rate-limits) | RPM, RPD, TPM, TPD: the quotas your service inherits from its provider |
| [LangSmith cost tracking](https://docs.langchain.com/langsmith/cost-tracking) | Where per-trace token cost comes from, for the cost budget |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langsmith>=0.14.11,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-groq", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-production-checklist'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · What "production" adds to an agent

In Project 09 the support desk worked: it answered, looked up invoices and issued refunds. Shipping it as a service adds questions the notebook never asked:

| Area | Notebook answer | Production question | Lesson |
|---|---|---|---|
| Reliability | "It worked when I ran it" | Which share of runs must succeed, and how do you know? | 12.01, 12.07 |
| Latency | "It feels fast" | First token within how many seconds, for which share of users? | 12.01, 12.03 |
| Cost | "A few cents" | Per request, per user per day, per day overall, and who stops the spend? | 12.01, 12.02 |
| Provider failures | Restart the kernel | Retry what, how often, fall back to what? | 12.02 |
| Concurrency | One run at a time | 20 users at once without blocking each other | 12.03 |
| Interface | `app.invoke(...)` | HTTP API, streaming, cancel, resume after a disconnect | 12.04 |
| State | `InMemorySaver` | Survives a restart; one user can't read another's thread | 12.05 |
| Packaging | Your laptop | A container anyone can run, with its database | 12.06 |
| Operations | Read the output | Traces, metrics, alerts, a runbook, a release gate | 12.07, 12.09 |

None of the rows is about the prompt. Most production work is everything around the model call.

In [ ]:
# 🧪 Offline: score the Project 09 desk as it is today (be honest; 0 = missing, 1 = partly, 2 = done).
readiness = {
    "SLOs written down": 0,
    "Retries and fallback for the model": 1,   # get_model sets max_retries=2, no fallback, no rules
    "Concurrency tested": 0,
    "HTTP API with streaming and cancel": 0,
    "Durable state and per-user isolation": 0,
    "Container and compose file": 0,
    "Tracing, metrics, alerts, runbook": 1,    # LangSmith tracing from Module 10, nothing else
    "Eval gate before release": 1,             # Module 10 evals exist; nothing blocks a release yet
    "Cost budget enforced": 0,
}
score = sum(readiness.values())
print(f"readiness {score}/{2 * len(readiness)}")
for item, value in readiness.items():
    print(f"  {'·○◐●'[value + 1]} {item}")

**You should now see** `readiness 3/18` and one line per item. By the end of Project 12 every line should be ●.

### What just happened?

You turned "is it production-ready?" into a list of concrete, checkable items. Each later lesson moves at least one of them to ●.

---
## Part 2 · SLIs and SLOs for an agent

An **SLI** (service level indicator) is a ratio: *good events ÷ valid events*. An **SLO** is a target for it over a window: "99% of runs in the last 7 days finish without a service error". For an agent, three SLIs cover what users notice:

| SLI | Good event | Valid events | Why it matters for agents |
|---|---|---|---|
| Availability | Run finished without a **service** error | Accepted runs (not rejected by quota) | Provider errors, tool crashes, timeouts |
| Time to first token (TTFT) | First token within 3 s | Runs that streamed | Agents think before they speak; users leave if nothing appears |
| Duration | Whole run within 60 s | Successful runs | Tool loops and slow tools hide here |

Two choices matter. A run the user **cancelled** is not a failure of the service, so it counts as good for availability. A run **rejected** by a quota never started, so it is not valid at all (it shows up in the cost budget instead, Part 5).

We need data to evaluate them. A real service has its own run log (Lesson 12.07 builds it); here `synthetic_runs` simulates one week of 500 runs from 25 users with a realistic shape: mostly fast, a small tail of slow queues and long tool loops, a few errors.

In [ ]:
# 🧪 Offline: one simulated week of runs (labelled simulated: not measurements of a real service).
from mastery.service import default_slos, evaluate_slo, synthetic_runs

runs = synthetic_runs(500, seed=12)
print("runs by status:", {s: sum(r.status == s for r in runs) for s in ("ok", "error", "cancelled", "rejected")})

slos = default_slos(ttft_s=3, duration_s=60)
results = [evaluate_slo(slo, runs) for slo in slos]
table([{"slo": r.name, "target": r.target, "good/valid": f"{r.good}/{r.eligible}",
        "ratio": round(r.ratio, 3), "95% interval": f"[{r.interval.low:.3f}, {r.interval.high:.3f}]",
        "met": r.met} for r in results],
      ["slo", "target", "good/valid", "ratio", "95% interval", "met"])

**You should now see** 487 ok, 4 errors, 5 cancelled and 4 rejected runs, then three rows: availability 0.992 (met), **TTFT 0.972 (not met)** and duration 0.994 (met).

### What just happened?

The service is up and runs finish in time, but almost 3% of users waited more than 3 s for the first token. That is the kind of problem nobody notices in a notebook. The interval matters too: TTFT's upper bound (0.983) is below 0.99, so this is not bad luck over a small sample.

---
## Part 3 · Error budgets, burn rate and a release policy

A 99% SLO allows 1% bad events: that allowance is the **error budget**. With 496 valid runs the availability budget is about 5 bad runs a week. You can spend it on risky releases, experiments or provider hiccups; when it is gone, reliability work comes first.

**Burn rate** says how fast you are spending it: the error rate in a recent window divided by the rate the SLO allows. Burn rate 1 uses the budget up exactly at the end of the window; the SRE workbook pages at **14.4** over one hour, which spends 2% of a 30-day budget in that hour.

In [ ]:
# 🧪 Offline: budget left, and the policy it implies.
from mastery.service import burn_rate, error_budget_policy

for r in results:
    print(f"{r.name:<13} budget {r.budget_used}/{r.budget_total:.1f} used, {r.budget_remaining:+.2f} left → {error_budget_policy(r)}")

# The same week with a provider slowdown in its last 10%: compare burn rates in that window.
incident = synthetic_runs(500, seed=12, incident=True)
since = 0.9 * 7 * 24 * 3600
print()
table([{"slo": s.name, "burn (normal)": round(burn_rate(s, runs, since=since), 1),
        "burn (incident)": round(burn_rate(s, incident, since=since), 1)} for s in slos],
      ["slo", "burn (normal)", "burn (incident)"])

**You should now see** availability `+0.19 left → caution`, TTFT `-1.82 left → freeze`, duration `normal`; then burn rates around 2 and 4 in the normal week and **8.2 (availability) and 67.3 (TTFT)** during the incident.

### What just happened?

- The policy is a rule agreed **in advance**: above half the budget, ship; above zero, only low-risk releases; below zero, freeze features. It ends arguments about "one more feature".
- A burn rate of 67 means the TTFT budget for a week is gone in a few hours: that pages someone. Burn rates in the 2–4 range at the end of an ordinary week are noise from a handful of runs; real alerting uses two windows (one hour and five minutes) so a short blip doesn't page.
- With little traffic, burn rates jump a lot. That's why the alert also needs a minimum number of events (Lesson 12.07).

---
## Part 4 · A latency budget: find the stage that breaks the SLO

"First token within 3 s" is shared by several stages: waiting for a worker (**queue**), the model's own TTFT, tool calls before the answer, and streaming the answer. A **latency budget** gives each stage its share, so each has an owner and a number. Compare stages at the **same percentile as the SLO**: a 99% SLO is about the slowest 1%.

In [ ]:
# 🧪 Offline: measured stage latency vs the budget, at p95 and at p99.
from mastery.service import LatencyBudget, percentile

budget = LatencyBudget({"queue": 0.5, "model_ttft": 2.0, "tools": 3.0, "stream": 10.0})
finished = [r for r in runs if r.stages]
for q in (95, 99):
    measured = {stage: percentile([r.stages[stage] for r in finished], q) for stage in budget.stages}
    print(f"p{q}:")
    table(budget.check(measured), ["stage", "budget_s", "measured_s", "ok"])

**You should now see** every stage `True` at p95, and at p99 **queue 3.38 s** (budget 0.5 s) and stream 51.9 s over budget.

### What just happened?

At p95 everything looks fine, which is why p95 dashboards miss 99% SLOs. At p99 the culprit for the TTFT miss is the **queue**, not the model: a few runs waited seconds for a free worker. That is a concurrency problem (blocking calls, too few workers), and Lesson 12.03 fixes it. The long `stream` tail is the duration SLO's business, and it still met its 95% target.

---
## Part 5 · A cost budget, enforced before the run

A cost budget you only read in a monthly bill is a report, not a budget. Enforce it **before** a run starts, with an estimate: the prompt you are about to send plus the output cap you set. Three levels, checked in order:

| Level | Protects against | Example |
|---|---|---|
| Per request | One huge prompt or runaway output | $0.002 |
| Per user per day | One user (or a leaked key) draining the service | $0.01 |
| Global per day: the **kill switch** | A bug or an attack across all users | $0.10 |

The numbers are tiny because `gpt-oss-20b` is cheap ($0.075 per 1M input tokens, $0.30 per 1M output at the price `mastery.pricing` verified); a 3,000-token prompt with a 300-token answer costs about $0.0003. On the free tier the real limit is Groq's tokens per day, and the same ledger protects it.

In [ ]:
# 🧪 Offline: replay a day of requests through the ledger (simulated users: one of them is a script).
import random

from mastery.service import CostBudget, SpendLedger, estimate_request_cost

MODEL = "openai/gpt-oss-20b"
ledger = SpendLedger(CostBudget(per_request=0.002, per_user_day=0.01, global_day=0.10))
rng = random.Random(1)
outcome = {}
for i in range(400):
    user = "u-script" if i % 3 == 0 else f"u{rng.randrange(20)}"         # a third of the traffic is one client
    prompt_tokens = 30_000 if i == 7 else rng.randint(1_500, 4_000)        # one oversized prompt
    estimate = estimate_request_cost(MODEL, prompt_tokens, 300)
    decision = ledger.check("2026-10-16", user, estimate)
    outcome[decision.reason] = outcome.get(decision.reason, 0) + 1
    if decision.allowed:
        ledger.charge("2026-10-16", user, estimate)                       # in a service: charge the real usage after the run

for reason, n in outcome.items():
    print(f"{n:>4}  {reason}")
print(f"spent ${ledger.spent('2026-10-16'):.4f}; top users:", [(u, round(c, 4)) for u, c in ledger.top_users("2026-10-16", 3)])

**You should now see** `297  ok`, one request rejected for its size (the 30,000-token prompt: estimate $0.0023 > cap $0.0020) and `102  user's daily budget used up`, all of them from `u-script`, which leads the top-users list at $0.01. Total spend is $0.0891, under the $0.10 kill switch.

### What just happened?

- The heavy client hit **its own** cap and everyone else kept working: per-user caps keep one tenant from becoming everyone's outage.
- The oversized prompt was refused before it cost anything. Return a clear error (HTTP 413 or 429 with a message), never a silent truncation.
- Estimates are upper bounds; charge the real `usage_metadata` after the run (Lesson 12.02 reads it) so the ledger matches the bill.
- Lower `global_day` to `0.05` and re-run: after 167 runs the kill switch refuses everyone (196 requests). That is the point of it: a bug costs at most one day's cap.

---
## Part 6 · Failure-mode review

Before writing service code, list what can fail, what the user sees, how you will know, and what limits the damage. This is a lightweight FMEA (failure mode and effects analysis); keep it next to the code and update it after every incident.

| Failure | User sees | Detect with | Limit the damage | Lesson |
|---|---|---|---|---|
| Provider 429 (rate limit) | Slow or failed answer | Error rate by `kind`; burn rate | Retry with backoff on transient errors only; client-side rate limiter | 12.02 |
| Provider outage / 5xx | Failed answer | Availability SLI | Fallback model; clear error after retries | 12.02 |
| Invalid key (401) | Every run fails | Error kind `auth`; alert at once | Don't retry; fail fast; page | 12.02 |
| Tool loop | Long wait, high cost | Duration SLI; model calls per run | Call limits; recursion limit | 12.02 |
| Slow queue | Late first token | Queue p99 | Async code, more workers, back-pressure | 12.03 |
| Client disconnects mid-stream | Lost answer | Resumes per run | Run continues server-side; resume with `Last-Event-ID` | 12.04 |
| Double submit | Two refunds | Duplicate idempotency keys | Idempotency key per side effect | 12.04, 12.05 |
| Restart mid-run | Thread lost | Runs without a final state | Postgres checkpointer; resume | 12.05 |
| One user reads another's thread | Data leak | Auth tests | Owner check on every thread | 12.05 |
| Prompt injection triggers a refund | Money lost | Refund log review; eval set | Human approval above a threshold (Module 08) | 12.05, 12.09 |
| Cost spike | Bill, or free-tier quota gone | Spend ledger; daily cost | Per-user caps; kill switch | 12.01 |

In [ ]:
# 🧪 Offline: rank the review by risk = likelihood × impact (1–3 each; your estimates, not measurements).
review = [
    ("Provider 429", 3, 2), ("Provider outage", 2, 3), ("Invalid key", 1, 3), ("Tool loop", 2, 2),
    ("Slow queue", 3, 2), ("Disconnect mid-stream", 3, 1), ("Double submit", 2, 3),
    ("Restart mid-run", 2, 2), ("Cross-user read", 1, 3), ("Injection triggers refund", 2, 3), ("Cost spike", 2, 2),
]
ranked = sorted(review, key=lambda row: -row[1] * row[2])
table([{"failure": f, "likelihood": l, "impact": i, "risk": l * i} for f, l, i in ranked],
      ["failure", "likelihood", "impact", "risk"])

**You should now see** risk 6 at the top: provider 429, provider outage, slow queue, double submit and injection-triggered refunds.

### What just happened?

The ranking is a conversation starter, not a calculation: the numbers are estimates. What it gives you is an **order of work**. Notice that the top items are the next lessons: provider failures (12.02), the queue (12.03) and side effects (12.04–12.05).

---
## Part 7 · 💸 Measure TTFT on the real model

Synthetic runs taught the method; now one real number. We stream three short answers from `gpt-oss-20b` and record the time to the first token with content and the total time, as `RunRecord`s, so the same SLO code evaluates them.

> 💸 Calls `gpt-oss-20b` (a few calls, a few thousand tokens of your free-tier quota). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
import time

from mastery.service import RunRecord

if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_model = get_model("fast", max_tokens=200)
    live_runs = []
    for i, question in enumerate(["What is an SLO, in one sentence?", "Name two causes of slow agents.", "What is a burn rate?"]):
        start, first = time.perf_counter(), None
        for chunk in live_model.stream(question):
            if first is None and chunk.content:
                first = time.perf_counter() - start
        total = time.perf_counter() - start
        live_runs.append(RunRecord(f"live{i}", "me", start, "ok", first, total))
        print(f"run {i}: TTFT {first if first is not None else float('nan'):.2f} s, total {total:.2f} s")
    ttft = evaluate_slo(slos[1], live_runs)
    print(f"TTFT SLO on {ttft.eligible} live runs: {ttft.good}/{ttft.eligible}, interval [{ttft.interval.low:.2f}, {ttft.interval.high:.2f}]")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** three runs, typically with TTFT well under a second on Groq even though `gpt-oss-20b` reasons before answering (reasoning tokens arrive first; the clock stops at the first **content** token), and an interval that runs from about 0.4 to 1.

### What just happened?

Three runs prove the code works, not that the SLO is met: the interval is far too wide. A real SLO needs hundreds of runs, which is why you measure it from the service's own log (Lesson 12.07) rather than from ad-hoc probes.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Dashboard green, users complain | p95 or averages watched for a 99% SLO | Look at the percentile the SLO names (Part 4) |
| Availability looks worse after adding quotas | Rejected runs counted as failures | Rejected runs are not valid events; track them in the cost budget |
| Burn-rate alert pages every night | Single short window, few events | Two windows (1 h and 5 min) and a minimum event count |
| SLO met, yet users are unhappy | Wrong SLI (e.g. duration when they care about TTFT) | Pick SLIs from what users notice; review them with support |
| Monthly bill surprises you | Cost read after the fact | Enforce the ledger before the run; per-user caps; kill switch |
| TTFT probe reports `nan` | The stream produced reasoning but no content (max_tokens too low) | Raise `max_tokens`, or count reasoning chunks if your UI shows them |

## ⚠️ Production notes

> ⚠️ **Production note: write the SLOs and the policy before launch, with whoever owns the product.**
> An SLO nobody agreed to is a wish. Write targets, windows, what counts as valid, and what happens when the budget runs out, and review them each quarter.

> ⚠️ **Production note: store the run log you need for SLOs from day one.** (Parts 2–4)
> Per run: user, start time, status, TTFT, duration, stage timings, tokens and cost. Without it you can't compute SLIs, burn rates or budgets later. Lesson 12.07 adds it to the service.

## 🏋️ Exercises

1. **(Easy)** Tighten TTFT to 2 s (`default_slos(ttft_s=2)`) and re-evaluate. Which policy applies now?
2. **(Medium)** Add a fourth SLO: "95% of successful runs cost at most $0.0005". Use `SLO(...)` with a `good` function that returns `None` for runs that don't count.
3. **(Medium)** In Part 5, give `u-script` its own higher cap (a "pro" plan) without raising everyone's. Sketch the change to `SpendLedger.check`.
4. **(Stretch)** Implement a two-window burn-rate alert: page when burn rate is above 14.4 over the last hour **and** over the last 5 minutes. Test it on `synthetic_runs(incident=True)` with a shorter `window_s` so the windows contain runs.

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
tight = evaluate_slo(default_slos(ttft_s=2)[1], runs)
print(round(tight.ratio, 3), error_budget_policy(tight))   # 0.958, still "freeze"
```

**2.**
```python
from mastery.service import SLO
cheap = SLO("cost", "95% of successful runs cost at most $0.0005", 0.95,
            lambda r: None if r.status != "ok" else r.cost_usd <= 0.0005)
print(evaluate_slo(cheap, runs).ratio)   # 1.0: the simulated runs all stay under $0.0005; try 0.0003
```

**3.** Keep a `{user: cap}` dict next to the budget and use `caps.get(user, budget.per_user_day)` in the per-user check. Plans belong in your user store, not in code; the ledger only reads them.

**4.**
```python
def page(slo, runs, now):
    long = burn_rate(slo, runs, since=now - 3600)
    short = burn_rate(slo, runs, since=now - 300)
    return long > 14.4 and short > 14.4
day = synthetic_runs(2000, seed=3, incident=True, window_s=6 * 3600)
print(page(slos[1], day, now=6 * 3600))   # True: burn rates about 45 (1 h) and 73 (5 min)
```
</details>

## 🔎 Docs drill (5 min)

In the SRE workbook's alerting chapter, what burn rate and window pair is recommended for a page, and what share of a 30-day budget has been spent when it fires?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) and find the table of recommended parameters.
2. The page row uses a burn rate of **14.4** over a one-hour long window (with a 5-minute short window).
3. The text explains that this fires once **2%** of the budget is spent (14.4 × 1 h ÷ 720 h ≈ 2%).

</details>

## ✅ Checkpoint

- [ ] I scored the Project 09 desk against a production checklist.
- [ ] I evaluated three SLOs with 95% intervals and explained why cancelled and rejected runs count the way they do.
- [ ] I computed error budget left and burn rate, and applied a release policy.
- [ ] I found the stage that breaks the TTFT SLO at p99 (the queue).
- [ ] I enforced per-request, per-user and global cost caps before runs.
- [ ] I ranked a failure-mode review and know which lesson addresses each item.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Groq rate limits](https://console.groq.com/docs/rate-limits) and log in [`docs/research-log.md`](../../docs/research-log.md) the free-tier RPM, RPD, TPM and TPD for `openai/gpt-oss-20b` today. Those quotas are the hard ceiling of every budget in this module.

## Summary

- Production adds reliability, latency, cost, concurrency, state, packaging and operations around the model call.
- SLI = good ÷ valid events; SLO = target over a window; report it with an interval.
- The error budget turns reliability into a release policy; burn rate tells you how fast it goes.
- Compare stages at the SLO's percentile: here the queue, not the model, breaks TTFT.
- Enforce cost before the run: per request, per user per day, and a global kill switch.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-02-resilience-cost-caching.ipynb** → [`12-02-resilience-cost-caching.ipynb`](12-02-resilience-cost-caching.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)